# PPC v10.2 — v8 Architecture + Improved Training
## What changed from v10.1 (failed PSK approach)
- **v8 MLP refinement stages** (proven 2.212mm) replace PSK (stuck at 10.9mm)
- **ALL 285/285 v8 weights loaded** — zero skips
- **Chamfer ramp**: OFF for ep1-16, linear 0→1 over ep16-41
- **Early stopping** patience=40, **LR=8e-5** matching v8
- No encoder freeze (v8 trained everything from ep1)


In [6]:
"""
PPC v10.2 — v8 Architecture + Improved Training
================================================
Same proven v8 architecture (MLP refinement stages) with training fixes:
  1. Load ALL v8 weights (285/285 — zero skips)
  2. Chamfer loss RAMP (v8's critical trick: gap-first, chamfer later)
  3. Early stopping patience=40 (v8 degraded 0.3mm after ep79)
  4. 128³ gap + confidence + Z-gap losses preserved
  5. PSK/VRCNet REMOVED (failed: stuck at 10.9mm for 50 epochs)
"""
import os, sys, json, time, math, random, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk

warnings.filterwarnings('ignore', category=FutureWarning)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    torch.cuda.set_per_process_memory_fraction(min(50.0/total_gb, 0.95), 0)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037')
SPLIT_FILE  = DATA_ROOT / 'dataset_split.json'
PROJECT_DIR = Path('/apps/app/pandu/ppc_network_v10_1')
CKPT_DIR    = PROJECT_DIR / 'checkpoints'
LOG_DIR     = PROJECT_DIR / 'logs'
RESULTS_DIR = PROJECT_DIR / 'results'
V8_CKPT_DIR = Path('/apps/app/pandu/ppc_network_v8/checkpoints')
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
IMG_SIZE, COARSE_VOL_SIZE, AUX_VOL_SIZE = 512, 32, 48
GAP_VOL_SIZE     = 128
N_POINTS         = 5120
ENC_CHANNELS     = 192
VOL_CHANNELS     = 128
DEC_CHANNELS     = 128
QUERY_DIM        = 256
N_REFINE_STAGES  = 3       # v8 used 3 MLP stages (proven 2.212mm)
PRETRAINED       = True
FREEZE_ENC_EPOCHS = 0      # no freeze — v8 trained everything from ep1

# v8 analysis: peaked ep79 then degraded. Match v8 LR=8e-5.
BATCH_SIZE, NUM_WORKERS, EPOCHS = 2, 4, 150
LR, WEIGHT_DECAY = 8e-5, 1e-5
WARMUP_EPOCHS    = 10       # v8 used 10 warmup epochs
USE_AMP, GRAD_CLIP = True, 1.0
EARLY_STOP_PATIENCE = 40   # v8 degraded 0.3mm after peak; stop earlier

# Chamfer ramp: v8's critical trick — train gap/spatial first, then add chamfer
# v8 schedule: ramp=0 for ep1-16, then 0.04/epoch → 1.0 at ep41
CHAMFER_RAMP_START  = 16   # start ramping chamfer at this epoch
CHAMFER_RAMP_END    = 41   # chamfer weight reaches 1.0 here

W_CHAMFER, W_GAP, W_AXIAL    = 1.0, 3.0, 0.6
W_SW, W_REPULSION, W_AUX_OCC = 0.3, 0.02, 0.05
W_CONF, W_ZGAP               = 0.30, 0.50
W_OFFSET, W_PROJ              = 0.0005, 0.02
MAX_EVAL = 103

CKPT_PATH    = CKPT_DIR / 'latest_checkpoint.pth'
BEST_CKPT    = CKPT_DIR / 'best_checkpoint.pth'
TRAINING_LOG = LOG_DIR  / 'training_log.json'

print('='*60)
print('CONFIG — PPC v10.2 (v8 arch + improved training)')
print('='*60)
for k,v in dict(STAGES=N_REFINE_STAGES, QUERY_DIM=QUERY_DIM,
                GAP=GAP_VOL_SIZE, W_GAP=W_GAP,
                LR=LR, RAMP=f'{CHAMFER_RAMP_START}-{CHAMFER_RAMP_END}',
                PATIENCE=EARLY_STOP_PATIENCE, EPOCHS=EPOCHS).items():
    print(f'  {k:<16} = {v}')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
CONFIG — PPC v10.2 (v8 arch + improved training)
  STAGES           = 3
  QUERY_DIM        = 256
  GAP              = 128
  W_GAP            = 3.0
  LR               = 8e-05
  RAMP             = 16-41
  PATIENCE         = 40
  EPOCHS           = 150


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def load_vtk_points(path):
    r = vtk.vtkPolyDataReader(); r.SetFileName(str(path)); r.Update()
    p = r.GetOutput(); return np.array([p.GetPoint(i) for i in range(p.GetNumberOfPoints())], np.float32)

def save_vtk_points(points, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for p in points: vp.InsertNextPoint(float(p[0]),float(p[1]),float(p[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)): vc.InsertNextCell(1); vc.InsertCellPoint(i)
    pd = vtk.vtkPolyData(); pd.SetPoints(vp); pd.SetVerts(vc)
    w = vtk.vtkPolyDataWriter(); w.SetFileName(str(path)); w.SetInputData(pd); w.SetFileTypeToASCII(); w.Write()

def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')
    if img.size != (size,size): img = img.resize((size,size), Image.BILINEAR)
    return np.array(img, np.float32) / 255.0

def load_projection_matrix(path):
    with open(path) as f: vals = [float(v) for v in f.read().split()]
    return np.array(vals, np.float32).reshape(3,4)

def load_split(p=SPLIT_FILE):
    with open(p) as f: d = json.load(f)
    return d['train'], d['val'], d['test']

def append_log(path, rec):
    log = {'records': []}
    if path.exists():
        with open(path) as f: log = json.load(f)
    log['records'].append(rec)
    tmp = path.with_suffix('.tmp')
    with open(tmp,'w') as f: json.dump(log, f, indent=2)
    tmp.replace(path)

def pts_to_local(pts, c, s): return (pts - c[None]) / (s[None] + 1e-6)
def local_to_world(pts, c, s): return pts * s[:,None,:] + c[:,None,:]
def compute_scale(gt):
    e = (gt.max(0)-gt.min(0)).astype(np.float32)
    s = e*0.55 + np.array([20.,20.,30.], np.float32)
    return np.maximum(s, np.array([50.,50.,80.], np.float32))

def make_aux_occ(pl, vs=AUX_VOL_SIZE, d=1):
    p = np.clip((np.asarray(pl,np.float32)+1)*0.5, 0, 0.999999)
    idx = np.clip(np.floor(p*vs).astype(np.int32), 0, vs-1)
    o = np.zeros((vs,)*3, np.float32)
    for dx in range(-d,d+1):
        for dy in range(-d,d+1):
            for dz in range(-d,d+1):
                o[np.clip(idx[:,0]+dx,0,vs-1),np.clip(idx[:,1]+dy,0,vs-1),np.clip(idx[:,2]+dz,0,vs-1)] = 1.0
    return o

def make_gap_occ(pl, vs=GAP_VOL_SIZE):
    p = np.clip((np.asarray(pl,np.float32)+1)*0.5, 0, 0.999999)
    idx = np.clip(np.floor(p*vs).astype(np.int32), 0, vs-1)
    o = np.zeros((vs,)*3, np.float32); o[idx[:,0],idx[:,1],idx[:,2]] = 1.0; return o

def flip_P(P, s=IMG_SIZE):
    F_mat = np.array([[-1,0,s-1],[0,1,0],[0,0,1]], np.float32); return F_mat @ P

class LumbarDataset(Dataset):
    def __init__(self, ids, root=DATA_ROOT, aug=False):
        self.ids, self.root, self.aug = ids, Path(root), aug
        self.norm = transforms.Normalize(mean=[0.485], std=[0.229])
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        pid = self.ids[i]; d = self.root/pid
        dap = load_drr_image(d/'AP_0'/'drr_AP_0.png')
        dlp = load_drr_image(d/'LP_90'/'drr_LP_90.png')
        Pap = load_projection_matrix(d/'AP_0'/'P_AP_0.txt')
        Plp = load_projection_matrix(d/'LP_90'/'P_LP_90.txt')
        gt = load_vtk_points(d/'gt_ppc.vtk')
        c = gt.mean(0).astype(np.float32); s = compute_scale(gt)
        gl = np.clip(pts_to_local(gt,c,s), -1, 1)
        ao = make_aux_occ(gl); go = make_gap_occ(gl)
        n = len(gt); sel = np.random.choice(n, N_POINTS, replace=(n<N_POINTS))
        gtw, gtl = gt[sel].astype(np.float32), gl[sel].astype(np.float32)
        if self.aug:
            for x in [dap,dlp]: x[:] = np.clip(x*(1+(np.random.rand()*2-1)*0.15), 0, 1)
            if np.random.rand() < 0.5:
                dap = dap[:,::-1].copy(); dlp = dlp[:,::-1].copy()
                Pap = flip_P(Pap); Plp = flip_P(Plp)
        return {'drr_ap': self.norm(torch.from_numpy(dap).unsqueeze(0).float()),
                'drr_lp': self.norm(torch.from_numpy(dlp).unsqueeze(0).float()),
                'P_ap': torch.from_numpy(Pap).float(), 'P_lp': torch.from_numpy(Plp).float(),
                'gt_ppc_world': torch.from_numpy(gtw).float(),
                'gt_ppc_local': torch.from_numpy(gtl).float(),
                'aux_occ': torch.from_numpy(ao).float(), 'gap_occ': torch.from_numpy(go).float(),
                'center': torch.from_numpy(c).float(), 'scale': torch.from_numpy(s).float(),
                'patient_id': pid}

train_ids, val_ids, test_ids = load_split()
print(f'Split: train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}')

Split: train=829 val=103 test=105


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL — v8 Encoder + Lightweight VRCNet Refinement (NO VAE)
# ══════════════════════════════════════════════════════════════════════════════

class Encoder2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=ENC_CHANNELS, pretrained=PRETRAINED):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        c1 = nn.Conv2d(in_ch,64,7,stride=2,padding=3,bias=False)
        if pretrained:
            with torch.no_grad(): c1.weight[:] = base.conv1.weight.mean(1,keepdim=True)
        base.conv1 = c1
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1, self.layer2, self.layer3 = base.layer1, base.layer2, base.layer3
        self.proj = nn.Conv2d(256, out_ch, 1)
    def forward(self, x): return self.proj(self.layer3(self.layer2(self.layer1(self.stem(x)))))

class FeatureLift(nn.Module):
    def __init__(self, in_ch=ENC_CHANNELS, out_ch=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        self.depth_embed = nn.Parameter(torch.randn(1,in_ch,depth,1,1)*0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch,out_ch,3,padding=1), nn.GroupNorm(8,out_ch), nn.GELU(),
            nn.Conv3d(out_ch,out_ch,3,padding=1), nn.GroupNorm(8,out_ch), nn.GELU())
    def forward(self, f2d):
        B,C,H,W = f2d.shape
        vol = f2d.unsqueeze(2).expand(B,C,COARSE_VOL_SIZE,H,W).contiguous()
        return self.refine(vol + self.depth_embed)

class BiplanarFusion(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, out_ch=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_ch*2,out_ch,1), nn.GroupNorm(8,out_ch), nn.GELU(),
            nn.Conv3d(out_ch,out_ch,3,padding=1), nn.GroupNorm(8,out_ch), nn.GELU())
    def forward(self, ap, lp):
        return self.fuse(torch.cat([ap.permute(0,1,4,2,3).contiguous(),
                                    lp.permute(0,1,2,4,3).contiguous()], 1))

class Block3D(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.block = nn.Sequential(nn.Conv3d(ic,oc,3,padding=1), nn.GroupNorm(8,oc), nn.GELU(),
                                    nn.Conv3d(oc,oc,3,padding=1), nn.GroupNorm(8,oc), nn.GELU())
    def forward(self, x): return self.block(x)

class CoarseUNet3D(nn.Module):
    def __init__(self, in_ch=VOL_CHANNELS, fc=DEC_CHANNELS):
        super().__init__()
        self.enc1 = Block3D(in_ch,96); self.down1 = nn.Conv3d(96,160,2,stride=2)
        self.enc2 = Block3D(160,160);  self.down2 = nn.Conv3d(160,224,2,stride=2)
        self.bottleneck = Block3D(224,224)
        self.up2 = nn.ConvTranspose3d(224,160,2,stride=2); self.dec2 = Block3D(320,160)
        self.up1 = nn.ConvTranspose3d(160,96,2,stride=2);  self.dec1 = Block3D(192,fc)
        self.aux_head = nn.Sequential(nn.Conv3d(fc,fc//2,3,padding=1), nn.GELU(), nn.Conv3d(fc//2,1,1))
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.down1(e1))
        b = self.bottleneck(self.down2(e2))
        d2 = self.dec2(torch.cat([self.up2(b),e2],1))
        d1 = self.dec1(torch.cat([self.up1(d2),e1],1))
        aux = F.interpolate(self.aux_head(d1), size=(AUX_VOL_SIZE,)*3, mode='trilinear', align_corners=True)
        return d1, aux

def project_points(P, pts, s=IMG_SIZE):
    B,N,_ = pts.shape
    h = torch.cat([pts, torch.ones(B,N,1,device=pts.device,dtype=pts.dtype)],-1)
    uvw = torch.bmm(h, P.transpose(1,2)); z = uvw[...,2:3].clamp(min=1e-6)
    uv = uvw[...,:2]/z; return uv, (uv/(s-1))*2-1, torch.log(z)

def sample_2d(fm, uvn):
    return F.grid_sample(fm, uvn.view(fm.shape[0],-1,1,2), mode='bilinear',
                         align_corners=True, padding_mode='border').squeeze(-1).transpose(1,2)

def sample_3d(vf, pl):
    g = torch.stack([pl[...,2],pl[...,1],pl[...,0]],-1).view(pl.shape[0],-1,1,1,3)
    return F.grid_sample(vf, g, mode='bilinear', align_corners=True,
                         padding_mode='border').squeeze(-1).squeeze(-1).transpose(1,2)

class QueryInit(nn.Module):
    def __init__(self, n=N_POINTS, fc=DEC_CHANNELS):
        super().__init__()
        grid = np.stack(np.meshgrid(np.linspace(-.8,.8,16), np.linspace(-.8,.8,16),
                                     np.linspace(-.9,.9,20), indexing='ij'),-1).reshape(-1,3).astype(np.float32)
        self.register_buffer('base', torch.from_numpy(grid)); self.n = n
        self.head = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Flatten(),
                                   nn.Linear(fc,256), nn.GELU(), nn.Linear(256,n*3))
    def forward(self, vf, aux=None):
        B = vf.shape[0]; off = self.head(vf).view(B,self.n,3)
        raw = self.base[None] + 0.20*torch.tanh(off)
        if aux is not None:
            gs = torch.stack([raw[...,2],raw[...,1],raw[...,0]],-1).view(B,self.n,1,1,3)
            gate = torch.sigmoid(F.grid_sample(aux, gs, mode='bilinear',
                                 align_corners=True, padding_mode='border').view(B,self.n)).unsqueeze(-1)
            raw = self.base[None] + gate*0.25*torch.tanh(off)
        return torch.clamp(raw, -1, 1)

# ── v8 MLP Refinement Stage (proven 2.212mm) ───────────────────────────────

class RefinementStage(nn.Module):
    """v8's proven MLP refinement: feature aggregation → MLP → offset."""
    def __init__(self, f2d=ENC_CHANNELS, f3d=DEC_CHANNELS, h=QUERY_DIM):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(f2d*2+f3d+3+2+2+1, h), nn.GELU(),
                                  nn.Linear(h, h), nn.GELU(), nn.Linear(h, 3))
    def forward(self, q, vf, fap, flp, Pap, Plp, c, s, aux=None):
        qw = local_to_world(q, c, s)
        _, ua, _ = project_points(Pap, qw); _, ul, _ = project_points(Plp, qw)
        B,N,_ = q.shape
        if aux is not None:
            gs = torch.stack([q[...,2],q[...,1],q[...,0]],-1).view(B,N,1,1,3)
            ov = F.grid_sample(aux, gs, mode='bilinear', align_corners=True,
                               padding_mode='border').view(B,N,1)
        else: ov = torch.zeros(B,N,1,device=q.device)
        x = torch.cat([sample_3d(vf,q), sample_2d(fap,ua), sample_2d(flp,ul), q, ua, ul, ov], -1)
        delta = 0.25 * torch.tanh(self.mlp(x))
        return q + delta, {'delta': delta}

class PPCNetV10_2(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_ap = Encoder2D(); self.encoder_lp = Encoder2D()
        self.lift_ap = FeatureLift(); self.lift_lp = FeatureLift()
        self.fusion = BiplanarFusion(); self.coarse3d = CoarseUNet3D()
        self.query_init = QueryInit()
        self.stages = nn.ModuleList([RefinementStage() for _ in range(N_REFINE_STAGES)])

    def forward(self, dap, dlp, Pap, Plp, c, s):
        fap = self.encoder_ap(dap); flp = self.encoder_lp(dlp)
        vap = self.lift_ap(fap); vlp = self.lift_lp(flp)
        fused = self.fusion(vap, vlp); vf, aux = self.coarse3d(fused)
        q = self.query_init(vf, aux)
        sa = []
        for stage in self.stages:
            q, a = stage(q, vf, fap, flp, Pap, Plp, c, s, aux)
            sa.append(a)
        q = torch.clamp(q, -1, 1)
        return {'pred_local': q, 'pred_world': local_to_world(q, c, s),
                'aux_occ_logits': aux, 'stage_aux': sa}

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
_m = PPCNetV10_2(); print(f'v10.2 params: {count_params(_m)/1e6:.1f} M'); del _m

v10.2 params: 21.8 M


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSSES — same as v10 but NO KL divergence
# ══════════════════════════════════════════════════════════════════════════════

def pairwise_sq(x,y):
    x2=(x**2).sum(-1,keepdim=True); y2=(y**2).sum(-1).unsqueeze(1)
    return (x2+y2-2*torch.bmm(x,y.transpose(1,2))).clamp_min(0)

def chamfer_loss(pred, gt, wp=1.0, wg=1.5, tr=15.0):
    d2=pairwise_sq(pred,gt)
    return wp*torch.clamp(d2.min(2)[0],max=tr**2).mean() + wg*d2.min(1)[0].mean()

def gap_penalty(pl, gap_occ):
    B,N,_=pl.shape; occ=gap_occ.unsqueeze(1)
    g=torch.stack([pl[...,2],pl[...,1],pl[...,0]],-1).view(B,N,1,1,3)
    s=F.grid_sample(occ,g,mode='bilinear',align_corners=True,padding_mode='border').view(B,N)
    return ((1.0-s).clamp(min=0)**2).mean()

def axial_density(pl, gl, nb=80, sig=0.020):
    c=torch.linspace(-1,1,nb,device=pl.device)
    pk=torch.exp(-0.5*((pl[...,2].unsqueeze(-1)-c)/sig)**2).sum(1)
    gk=torch.exp(-0.5*((gl[...,2].unsqueeze(-1)-c)/sig)**2).sum(1)
    pd=pk/(pk.sum(1,keepdim=True)+1e-8); gd=gk/(gk.sum(1,keepdim=True)+1e-8)
    return (pd-gd).abs().sum(1).mean()

def sw_loss(pred, gt, np_=50):
    dirs=F.normalize(torch.randn(np_,3,device=pred.device),-1)
    pp=torch.einsum('bnd,pd->bnp',pred,dirs); gp=torch.einsum('bnd,pd->bnp',gt,dirs)
    return ((pp.sort(1)[0]-gp.sort(1)[0])**2).mean()

def repulsion(pts, k=8, hxy=0.020, hz=0.040):
    d2=pairwise_sq(pts,pts); B,N,_=d2.shape
    d2=d2.masked_fill(torch.eye(N,device=pts.device,dtype=torch.bool).unsqueeze(0), float('inf'))
    ki=d2.topk(k,dim=-1,largest=False).indices
    pe=pts.unsqueeze(2).expand(B,N,k,3)
    nb=torch.gather(pts.unsqueeze(1).expand(B,N,N,3),2,ki.unsqueeze(-1).expand(B,N,k,3))
    df=pe-nb; da=(df[...,0]**2+df[...,1]**2)/(hxy**2)+df[...,2]**2/(hz**2)
    return torch.exp(-da).mean()

def conf_loss(cl, pl, gap_occ):
    B,N,_=pl.shape; occ=gap_occ.unsqueeze(1)
    g=torch.stack([pl[...,2],pl[...,1],pl[...,0]],-1).view(B,N,1,1,3)
    bl=F.grid_sample(occ,g,mode='bilinear',align_corners=True,padding_mode='border').view(B,N)
    return F.binary_cross_entropy_with_logits(cl, bl.detach())

def zgap_loss(pl, gl, ns=100, sig=0.015):
    c=torch.linspace(-1,1,ns,device=pl.device)
    gk=torch.exp(-0.5*((gl[...,2].unsqueeze(-1)-c)/sig)**2).sum(1)
    gd=gk/(gk.max(1,keepdim=True)[0]+1e-8); gm=(gd<0.05).float()
    pz=pl[...,2]; pi=((pz+1)*0.5*(ns-1)).long().clamp(0,ns-1)
    return torch.gather(gm,1,pi).mean()

def proj_loss(pw, gw, Pap, Plp):
    def ch2d(a,b): d2=pairwise_sq(a,b); return 0.5*(d2.min(2)[0].mean()+d2.min(1)[0].mean())
    pa,_,_=project_points(Pap,pw); ga,_,_=project_points(Pap,gw)
    pl,_,_=project_points(Plp,pw); gl,_,_=project_points(Plp,gw)
    return ch2d(pa,ga)+ch2d(pl,gl)

def dice_loss(logits, tgt, eps=1e-6):
    p=torch.sigmoid(logits).reshape(logits.shape[0],-1); t=tgt.reshape(tgt.shape[0],-1)
    return 1.0-((2*(p*t).sum(1)+eps)/(p.sum(1)+t.sum(1)+eps)).mean()

def chamfer_ramp(epoch):
    """v8's proven schedule: 0 for ep<16, linear 0→1 over ep16-41, then 1.0"""
    if epoch < CHAMFER_RAMP_START: return 0.0
    if epoch >= CHAMFER_RAMP_END: return 1.0
    return (epoch - CHAMFER_RAMP_START) / (CHAMFER_RAMP_END - CHAMFER_RAMP_START)

def total_loss(out, batch, epoch=0):
    pw,pl=out['pred_world'],out['pred_local']
    gw,gl=batch['gt_ppc_world'],batch['gt_ppc_local']
    lc=chamfer_loss(pw,gw); lg=gap_penalty(pl,batch['gap_occ'])
    la=axial_density(pl,gl); ls=sw_loss(pl,gl); lr=repulsion(pl)
    lz=zgap_loss(pl,gl)
    laux=(F.binary_cross_entropy_with_logits(out['aux_occ_logits'].squeeze(1),batch['aux_occ'])
          +dice_loss(out['aux_occ_logits'].squeeze(1),batch['aux_occ']))
    lo=torch.stack([a['delta'].abs().mean() for a in out['stage_aux']]).mean()
    wp=W_PROJ+max(0.0,(30-epoch)/30.0)*0.08
    lp=proj_loss(pw,gw,batch['P_ap'],batch['P_lp'])
    # Chamfer ramp: v8's critical trick — no chamfer for first 16 epochs
    ramp = chamfer_ramp(epoch)
    wc = W_CHAMFER * ramp
    loss=(wc*lc+W_GAP*lg+W_AXIAL*la+W_SW*ls+W_REPULSION*lr
          +W_ZGAP*lz+W_AUX_OCC*laux+W_OFFSET*lo+wp*lp)
    bd={k:float(v.detach().cpu()) for k,v in
        dict(total=loss,chamfer=lc,gap=lg,axial=la,sw=ls,
             zgap=lz,repulsion=lr,aux_occ=laux,offset=lo,proj=lp,ramp=torch.tensor(ramp)).items()}
    return loss, bd

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING — with v8 pretrained encoder loading + freezing
# ══════════════════════════════════════════════════════════════════════════════

def collate_fn(b):
    out = {}
    for k in b[0]:
        vals = [x[k] for x in b]
        out[k] = torch.stack(vals,0) if isinstance(vals[0], torch.Tensor) else vals
    return out

def to_dev(b):
    return {k:(v.to(device,non_blocking=True) if isinstance(v,torch.Tensor) else v) for k,v in b.items()}

def chamfer_np(pred, gt):
    pred,gt = np.asarray(pred,np.float32), np.asarray(gt,np.float32)
    d = np.linalg.norm(pred[:,None]-gt[None], axis=-1)
    dp,dg = d.min(1), d.min(0)
    def fs(th): p,r=(dp<=th).mean(),(dg<=th).mean(); return 2*p*r/(p+r) if (p+r)>0 else 0.0
    return {'chamfer_mm':float(.5*(dp.mean()+dg.mean())), 'fscore_1mm':float(fs(1)),
            'fscore_2mm':float(fs(2)), 'fscore_5mm':float(fs(5)),
            'hausdorff_95':float(max(np.percentile(dp,95),np.percentile(dg,95)))}

model = PPCNetV10_2().to(device)

# ── LOAD V8 PRETRAINED ENCODER ───────────────────────────────────────────────
v8_ckpt = V8_CKPT_DIR / 'best_chamfer_checkpoint.pth'
if not v8_ckpt.exists(): v8_ckpt = V8_CKPT_DIR / 'best_checkpoint.pth'
if not v8_ckpt.exists(): v8_ckpt = V8_CKPT_DIR / 'latest_checkpoint.pth'

if v8_ckpt.exists():
    print(f'Loading v8 pretrained weights from {v8_ckpt.name}...')
    v8_state = torch.load(v8_ckpt, map_location=device, weights_only=False)['model']
    # Map v8 keys → v10.1 keys (some modules were renamed)
    key_map = {
        'query_init.base_queries': 'query_init.base',
        'query_init.global_head.2.weight': 'query_init.head.2.weight',
        'query_init.global_head.2.bias': 'query_init.head.2.bias',
        'query_init.global_head.4.weight': 'query_init.head.4.weight',
        'query_init.global_head.4.bias': 'query_init.head.4.bias',
    }
    own = model.state_dict()
    loaded, skipped = 0, []
    for k, v in v8_state.items():
        target_k = key_map.get(k, k)  # remap if needed
        if target_k in own and own[target_k].shape == v.shape:
            own[target_k] = v; loaded += 1
        else:
            skipped.append(k)
    model.load_state_dict(own)
    print(f'  Loaded {loaded}/{len(v8_state)} v8 weights')
    if skipped:
        print(f'  Skipped {len(skipped)}: {skipped[:5]}...')
else:
    print('WARNING: No v8 checkpoint found, training from scratch!')

print(f'Model params: {count_params(model)/1e6:.1f} M')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type=='cuda')

train_ds = LumbarDataset(train_ids, aug=True)
val_ds = LumbarDataset(val_ids, aug=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_fn, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        collate_fn=collate_fn, persistent_workers=True)
print(f'Train: {len(train_ds)} -> {len(train_loader)} batches')
print(f'Val  : {len(val_ds)} -> {len(val_loader)} batches')

def save_ckpt(path, ep, best, hist):
    tmp = path.with_suffix('.tmp')
    torch.save({'model':model.state_dict(), 'optimizer':optimizer.state_dict(),
                'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict() if scaler.is_enabled() else None,
                'epoch':ep, 'best_chamfer':best, 'history':hist,
                'config':{'version':'v10.2_v8arch'}}, tmp)
    tmp.replace(path); print(f'  Saved -> {path.name} (ep {ep+1})')

def load_ckpt(path):
    st = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(st['model']); optimizer.load_state_dict(st['optimizer'])
    if st.get('scheduler'): scheduler.load_state_dict(st['scheduler'])
    if scaler.is_enabled() and st.get('scaler'): scaler.load_state_dict(st['scaler'])
    ep=st['epoch']+1; bc=st.get('best_chamfer',float('inf')); h=st.get('history',[])
    print(f'  Resumed from ep {ep} | best={bc:.3f}mm'); return ep, bc, h

start_epoch, best_chamfer, history, no_improve = 0, float('inf'), [], 0
if CKPT_PATH.exists():
    print(f'Found checkpoint: {CKPT_PATH}')
    start_epoch, best_chamfer, history = load_ckpt(CKPT_PATH)
else:
    print('No v10.2 checkpoint — starting with v8 weights.')

def set_encoder_frozen(frozen):
    for name, p in model.named_parameters():
        if any(x in name for x in ['encoder_ap','encoder_lp','lift_ap','lift_lp','fusion','coarse3d']):
            p.requires_grad_(not frozen)

print(f'\n{"="*60}')
print(f'Starting v10.2 training from epoch {start_epoch+1}...')
if FREEZE_ENC_EPOCHS > 0:
    print(f'  Encoder frozen for first {FREEZE_ENC_EPOCHS} epochs')
print(f'{"="*60}\n')

for epoch in range(start_epoch, EPOCHS):
    # Freeze/unfreeze encoder
    if epoch < FREEZE_ENC_EPOCHS:
        set_encoder_frozen(True)
        if epoch == 0: print('  [Encoder FROZEN]')
    elif epoch == FREEZE_ENC_EPOCHS:
        set_encoder_frozen(False)
        print('  [Encoder UNFROZEN — fine-tuning]')

    model.train(); t0=time.time(); acc={}; nb=0
    for bi, batch in enumerate(train_loader, 1):
        batch = to_dev(batch)
        # Warmup LR
        if epoch < WARMUP_EPOCHS:
            lr_s = max(0.1, (epoch*len(train_loader)+bi)/(WARMUP_EPOCHS*len(train_loader)))
            for pg in optimizer.param_groups: pg['lr'] = scheduler.get_last_lr()[0]*lr_s

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(batch['drr_ap'],batch['drr_lp'],batch['P_ap'],batch['P_lp'],
                       batch['center'],batch['scale'])
            loss, bd = total_loss(out, batch, epoch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        for k,v in bd.items(): acc[k]=acc.get(k,0)+v
        nb += 1
        if bi%50==0 or bi==len(train_loader):
            rmp = chamfer_ramp(epoch)
            print(f"  [Ep {epoch+1:3d}/{EPOCHS}] {bi:3d}/{len(train_loader)} "
                  f"ch={acc.get('chamfer',0)/nb:.4f} gap={acc.get('gap',0)/nb:.4f} "
                  f"ax={acc.get('axial',0)/nb:.4f} ramp={rmp:.2f}")
    scheduler.step()
    elapsed = time.time()-t0

    # Validation
    model.eval(); metrics=[]; nd=0
    with torch.no_grad():
        for batch in val_loader:
            if nd>=MAX_EVAL: break
            batch = to_dev(batch)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(batch['drr_ap'],batch['drr_lp'],batch['P_ap'],batch['P_lp'],
                           batch['center'],batch['scale'])
            pw=out['pred_world'].cpu().numpy(); gw=batch['gt_ppc_world'].cpu().numpy()
            for b in range(pw.shape[0]):
                if nd>=MAX_EVAL: break
                metrics.append(chamfer_np(pw[b],gw[b])); nd+=1

    if metrics:
        mc=np.mean([m['chamfer_mm'] for m in metrics])
        f1=np.mean([m['fscore_1mm'] for m in metrics])
        f2=np.mean([m['fscore_2mm'] for m in metrics])
        f5=np.mean([m['fscore_5mm'] for m in metrics])
        hd=np.mean([m['hausdorff_95'] for m in metrics])
        print(f"[Ep {epoch+1}] {elapsed:.0f}s Ch={mc:.3f}mm F@1={f1:.3f} F@2={f2:.3f} F@5={f5:.3f} HD95={hd:.3f}")
        rec={'epoch':epoch+1,'chamfer_mm':mc,'f1':f1,'f2':f2,'f5':f5,'hd95':hd,
             'train_gap':acc.get('gap',0)/max(1,nb)}
        history.append(rec); append_log(TRAINING_LOG, rec)
        save_ckpt(CKPT_PATH, epoch, best_chamfer, history)
        if mc < best_chamfer:
            best_chamfer = mc
            no_improve = 0
            save_ckpt(BEST_CKPT, epoch, best_chamfer, history)
            print(f"  ✓ New best: {best_chamfer:.3f} mm")
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE and epoch >= CHAMFER_RAMP_END:
                print(f"  ⛔ Early stop: no improvement for {no_improve} epochs")
                break

print(f'\nDone. Best Chamfer: {best_chamfer:.3f} mm')

Loading v8 pretrained weights from best_chamfer_checkpoint.pth...
  Loaded 285/285 v8 weights
Model params: 21.8 M
Train: 829 -> 415 batches
Val  : 103 -> 52 batches
Found checkpoint: /apps/app/pandu/ppc_network_v10_1/checkpoints/latest_checkpoint.pth
  Resumed from ep 50 | best=2.168mm

Starting v10.2 training from epoch 51...

  [Ep  51/150]  50/415 ch=0.0991 gap=0.9095 ax=0.0435 ramp=1.00
  [Ep  51/150] 100/415 ch=0.1003 gap=0.9104 ax=0.0436 ramp=1.00
  [Ep  51/150] 150/415 ch=0.0978 gap=0.9103 ax=0.0443 ramp=1.00
  [Ep  51/150] 200/415 ch=0.1091 gap=0.9106 ax=0.0443 ramp=1.00
  [Ep  51/150] 250/415 ch=0.0873 gap=0.9094 ax=0.0437 ramp=1.00
  [Ep  51/150] 300/415 ch=0.1000 gap=0.9097 ax=0.0439 ramp=1.00
  [Ep  51/150] 350/415 ch=0.1018 gap=0.9096 ax=0.0441 ramp=1.00
  [Ep  51/150] 400/415 ch=0.0953 gap=0.9091 ax=0.0440 ramp=1.00
  [Ep  51/150] 415/415 ch=0.0919 gap=0.9093 ax=0.0441 ramp=1.00
[Ep 51] 58s Ch=2.170mm F@1=0.117 F@2=0.542 F@5=0.964 HD95=4.716
  Saved -> latest_checkpoint.

  [Ep  62/150] 250/415 ch=0.1111 gap=0.9091 ax=0.0445 ramp=1.00
  [Ep  62/150] 300/415 ch=0.0970 gap=0.9088 ax=0.0445 ramp=1.00
  [Ep  62/150] 350/415 ch=0.0999 gap=0.9097 ax=0.0446 ramp=1.00
  [Ep  62/150] 400/415 ch=0.0904 gap=0.9094 ax=0.0444 ramp=1.00
  [Ep  62/150] 415/415 ch=0.0922 gap=0.9096 ax=0.0445 ramp=1.00
[Ep 62] 32s Ch=2.171mm F@1=0.117 F@2=0.542 F@5=0.964 HD95=4.716
  Saved -> latest_checkpoint.pth (ep 62)
  [Ep  63/150]  50/415 ch=0.0000 gap=0.9129 ax=0.0451 ramp=1.00
  [Ep  63/150] 100/415 ch=0.0904 gap=0.9113 ax=0.0459 ramp=1.00
  [Ep  63/150] 150/415 ch=0.0804 gap=0.9109 ax=0.0450 ramp=1.00
  [Ep  63/150] 200/415 ch=0.0651 gap=0.9105 ax=0.0444 ramp=1.00
  [Ep  63/150] 250/415 ch=0.0579 gap=0.9108 ax=0.0441 ramp=1.00
  [Ep  63/150] 300/415 ch=0.0676 gap=0.9107 ax=0.0440 ramp=1.00
  [Ep  63/150] 350/415 ch=0.0740 gap=0.9102 ax=0.0441 ramp=1.00
  [Ep  63/150] 400/415 ch=0.0842 gap=0.9098 ax=0.0443 ramp=1.00
  [Ep  63/150] 415/415 ch=0.0928 gap=0.9096 ax=0.0443 ramp=1.00

  [Ep  74/150] 300/415 ch=0.0662 gap=0.9097 ax=0.0445 ramp=1.00
  [Ep  74/150] 350/415 ch=0.0835 gap=0.9092 ax=0.0444 ramp=1.00
  [Ep  74/150] 400/415 ch=0.0959 gap=0.9095 ax=0.0444 ramp=1.00
  [Ep  74/150] 415/415 ch=0.0924 gap=0.9095 ax=0.0443 ramp=1.00
[Ep 74] 51s Ch=2.173mm F@1=0.117 F@2=0.541 F@5=0.964 HD95=4.721
  Saved -> latest_checkpoint.pth (ep 74)
  [Ep  75/150]  50/415 ch=0.0572 gap=0.9068 ax=0.0433 ramp=1.00
  [Ep  75/150] 100/415 ch=0.1104 gap=0.9091 ax=0.0446 ramp=1.00
  [Ep  75/150] 150/415 ch=0.1249 gap=0.9094 ax=0.0451 ramp=1.00
  [Ep  75/150] 200/415 ch=0.1395 gap=0.9087 ax=0.0451 ramp=1.00
  [Ep  75/150] 250/415 ch=0.1178 gap=0.9090 ax=0.0449 ramp=1.00
  [Ep  75/150] 300/415 ch=0.1023 gap=0.9090 ax=0.0446 ramp=1.00
  [Ep  75/150] 350/415 ch=0.0904 gap=0.9093 ax=0.0444 ramp=1.00
  [Ep  75/150] 400/415 ch=0.0913 gap=0.9092 ax=0.0444 ramp=1.00
  [Ep  75/150] 415/415 ch=0.0937 gap=0.9095 ax=0.0444 ramp=1.00
[Ep 75] 45s Ch=2.169mm F@1=0.117 F@2=0.543 F@5=0.964 HD95=4.715

  [Ep  86/150] 350/415 ch=0.0934 gap=0.9099 ax=0.0444 ramp=1.00
  [Ep  86/150] 400/415 ch=0.0947 gap=0.9099 ax=0.0444 ramp=1.00
  [Ep  86/150] 415/415 ch=0.0913 gap=0.9097 ax=0.0444 ramp=1.00
[Ep 86] 32s Ch=2.171mm F@1=0.117 F@2=0.542 F@5=0.964 HD95=4.712
  Saved -> latest_checkpoint.pth (ep 86)
  [Ep  87/150]  50/415 ch=0.0573 gap=0.9108 ax=0.0466 ramp=1.00
  [Ep  87/150] 100/415 ch=0.1371 gap=0.9105 ax=0.0455 ramp=1.00
  [Ep  87/150] 150/415 ch=0.0914 gap=0.9109 ax=0.0453 ramp=1.00
  [Ep  87/150] 200/415 ch=0.0982 gap=0.9105 ax=0.0452 ramp=1.00
  [Ep  87/150] 250/415 ch=0.0786 gap=0.9104 ax=0.0448 ramp=1.00
  [Ep  87/150] 300/415 ch=0.0776 gap=0.9100 ax=0.0447 ramp=1.00
  [Ep  87/150] 350/415 ch=0.0919 gap=0.9100 ax=0.0447 ramp=1.00
  [Ep  87/150] 400/415 ch=0.0897 gap=0.9099 ax=0.0445 ramp=1.00
  [Ep  87/150] 415/415 ch=0.0925 gap=0.9099 ax=0.0445 ramp=1.00
[Ep 87] 32s Ch=2.170mm F@1=0.117 F@2=0.542 F@5=0.964 HD95=4.716
  Saved -> latest_checkpoint.pth (ep 87)
  [Ep  88/150]  50/415

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST
# ══════════════════════════════════════════════════════════════════════════════
print('\nTest evaluation...')
if BEST_CKPT.exists():
    st=torch.load(BEST_CKPT,map_location=device,weights_only=False)
    model.load_state_dict(st['model']); print(f'  Loaded ep {st["epoch"]+1}')
model.eval()
test_ds=LumbarDataset(test_ids,aug=False)
test_loader=DataLoader(test_ds,batch_size=2,shuffle=False,num_workers=NUM_WORKERS,
                       pin_memory=True,collate_fn=collate_fn)
all_m=[]
with torch.no_grad():
    for batch in test_loader:
        batch=to_dev(batch)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out=model(batch['drr_ap'],batch['drr_lp'],batch['P_ap'],batch['P_lp'],
                     batch['center'],batch['scale'])
        pw=out['pred_world'].cpu().numpy(); gt=batch['gt_ppc_world'].cpu().numpy()
        pids=batch['patient_id']
        for b in range(pw.shape[0]):
            m=chamfer_np(pw[b],gt[b]); m['patient_id']=pids[b]; all_m.append(m)
            save_vtk_points(pw[b], RESULTS_DIR/f'{pids[b]}_pred.vtk')

print(f'\n{"="*60}\nTEST ({len(all_m)} patients)\n{"="*60}')
for k,lbl in [('chamfer_mm','Chamfer'),('fscore_1mm','F@1'),('fscore_2mm','F@2'),
              ('fscore_5mm','F@5'),('hausdorff_95','HD95')]:
    vals=[m[k] for m in all_m]
    print(f'  {lbl:<10} mean={np.mean(vals):.3f} std={np.std(vals):.3f}')

import csv
csv_path=RESULTS_DIR/'test_results.csv'
with open(csv_path,'w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['patient_id','chamfer_mm','fscore_1mm','fscore_2mm','fscore_5mm','hausdorff_95'])
    w.writeheader(); w.writerows(all_m)
print(f'Saved to {csv_path}')


Test evaluation...
  Loaded ep 10

TEST (105 patients)
  Chamfer    mean=2.450 std=1.551
  F@1        mean=0.111 std=0.050
  F@2        mean=0.514 std=0.141
  F@5        mean=0.944 std=0.096
  HD95       mean=6.051 std=7.423
Saved to /apps/app/pandu/ppc_network_v10_1/results/test_results.csv
